# Is the ÖBB On Time?
## GTFS Data Loading, Initial Exploration and Schedule Preparation

- Load the GTFS files
- Understand the structure of the timetable data
- Explore key entities (stops, trips, routes, stop times)
- Prepare the data for building a unified schedule dataset

## 1. Setup and GTFS folder check

First, we import the required libraries and define the path to the raw GTFS data folder.
Then we list all files in the folder to verify that the GTFS dataset was downloaded and extracted correctly.

In [ ]:
import pandas as pd
from pathlib import Path

GTFS_PATH = Path("../data/raw/gtfs")

print("Files we have in the GTFS folder:")
for f in GTFS_PATH.iterdir():
    print(f.name)

Files in GTFS folder:
agency.txt
.DS_Store
calendar_dates.txt
GTFS_Fahrplan_2026
stop_times.txt
levels.txt
shapes.txt
trips.txt
pathways.txt
stops.txt
calendar.txt
routes.txt


## 2. Load core GTFS tables

For the first stage, we load four main GTFS tables:

- `stops.txt`: station and stop information, including coordinates
- `stop_times.txt`: scheduled arrival and departure times for each stop of each trip
- `trips.txt`: individual train trips and their route identifiers
- `routes.txt`: route-level information, such as line names

These tables are enough to build the initial scheduled timetable dataset.

In [ ]:
stops = pd.read_csv(GTFS_PATH / "stops.txt")
stop_times = pd.read_csv(GTFS_PATH / "stop_times.txt")
trips = pd.read_csv(GTFS_PATH / "trips.txt")
routes = pd.read_csv(GTFS_PATH / "routes.txt")

print("stops:", stops.shape)
print("stop_times:", stop_times.shape)
print("trips:", trips.shape)
print("routes:", routes.shape)

stops: (7698, 9)
stop_times: (174124, 9)
trips: (15165, 8)
routes: (273, 5)


The loaded GTFS tables have the following size:

- `stops`: 7,698 rows and 9 columns
- `stop_times`: 174,124 rows and 9 columns
- `trips`: 15,165 rows and 8 columns
- `routes`: 273 rows and 5 columns

The largest table is `stop_times`, because it contains one row for each scheduled stop within each train trip.

## 3. Inspect table columns and size

Next, we inspect the column names to understand how the tables can be joined.

Important keys:
- `stop_id` connects `stop_times` with `stops`
- `trip_id` connects `stop_times` with `trips`
- `route_id` connects `trips` with `routes`

In [ ]:
print("stops columns:")
print(stops.columns.tolist())

print("\nstop_times columns:")
print(stop_times.columns.tolist())

print("\ntrips columns:")
print(trips.columns.tolist())

print("\nroutes columns:")
print(routes.columns.tolist())

stops columns:
['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'zone_id', 'location_type', 'parent_station', 'level_id', 'platform_code']

stop_times columns:
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'shape_dist_traveled']

trips columns:
['route_id', 'service_id', 'trip_id', 'shape_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'block_id']

routes columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type']


In [ ]:
stops.head()

,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
0,at:41:3087:0:1,Bad Sauerbrunn Bahnhof,47.773659,16.325066,NaN,NaN,Pat:41:3087,Level 0,1
1,at:41:3087:0:2,Bad Sauerbrunn Bahnhof,47.773647,16.325057,NaN,NaN,Pat:41:3087,Level 0,2
2,at:41:3087:2,Bus,47.774226,16.324509,NaN,NaN,Pat:41:3087,Level 0,NaN
3,at:41:3087:3,Zugang,47.773949,16.324356,NaN,2.0,Pat:41:3087,Level 0,NaN
4,at:41:3087:4,P+R,47.772578,16.325695,NaN,3.0,Pat:41:3087,Level 0,NaN


In [ ]:
stop_times.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled
0,1.TA.20-SV1-A-j26-1.1.R,05:27:00,05:27:00,at:44:44900:0:22,1,NaN,0,0,0.00
1,1.TA.20-SV1-A-j26-1.1.R,05:32:00,05:32:00,at:44:44886:0:2,2,NaN,0,0,4601.58
2,1.TA.20-SV1-A-j26-1.1.R,05:37:00,05:37:00,at:44:44880:0:32,3,NaN,0,0,5546.31
3,1.TA.20-SV1-B-j26-1.1.R,15:41:00,15:41:00,at:44:43649:0:22,1,NaN,0,0,0.00
4,1.TA.20-SV1-B-j26-1.1.R,15:47:00,15:47:00,at:44:43640:0:22,2,NaN,0,0,2389.60


In [ ]:
trips.head()

,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,10-A10-1-j26-1,TA+he100,1.TA.10-A10-1-j26-1.1.R,10-A10-1-j26-1.1.R,Salzburg Hauptbahnhof,RJ 658,1,725.0
1,10-A10-1-j26-1,TA+f0,10.TA.10-A10-1-j26-1.7.H,10-A10-1-j26-1.7.H,Flughafen Wien Bahnhof,RJ 551,0,732.0
2,10-A10-1-j26-1,TA+cm,11.TA.10-A10-1-j26-1.8.H,10-A10-1-j26-1.8.H,Wien Hauptbahnhof,D 1130,0,NaN
3,10-A10-1-j26-1,TA+2C,2.TA.10-A10-1-j26-1.2.R,10-A10-1-j26-1.2.R,Salzburg Hauptbahnhof,RJ 110,1,726.0
4,10-A10-1-j26-1,TA+ie100,3.TA.10-A10-1-j26-1.3.R,10-A10-1-j26-1.3.R,Salzburg Hauptbahnhof,RJ 658,1,725.0


In [ ]:
routes.head()

,route_id,agency_id,route_short_name,route_long_name,route_type
0,10-A10-1-j26-1,1,A10-1,Salzburg - Villach - Klagenfurt,2
1,10-A11-j26-1,1,A11,Wien - Salzburg - Kufstein - Innsbruck - Blude...,2
2,10-A12-j26-1,1,A12,Breclav - Wien - Wr.Neustadt - Mürzzuschlag - ...,2
3,10-A13-j26-1,1,A13,Wien - Salzburg - Kufstein - Innsbruck - Blude...,2
4,10-A14-j26-1,1,A14,Wien - Salzburg - Kufstein - Innsbruck - Blude...,2


## 4. Check join keys

Before merging the tables, we check whether the key columns contain missing values.

This is important because the merge depends on the following identifiers:

- `trip_id`: connects `stop_times` with `trips`
- `stop_id`: connects `stop_times` with `stops`
- `route_id`: connects `trips` with `routes`

In [ ]:
print("stop_times trip_id missing:", stop_times["trip_id"].isna().sum())
print("stop_times stop_id missing:", stop_times["stop_id"].isna().sum())
print("trips trip_id missing:", trips["trip_id"].isna().sum())
print("stops stop_id missing:", stops["stop_id"].isna().sum())
print("routes route_id missing:", routes["route_id"].isna().sum())

stop_times trip_id missing: 0
stop_times stop_id missing: 0
trips trip_id missing: 0
stops stop_id missing: 0
routes route_id missing: 0


All key columns contain 0 missing values.

This means that the main GTFS tables can be merged without losing rows because of missing join identifiers.

## 5. Build stop-level schedule dataset

Now we merge the GTFS tables into one schedule dataset.

The final dataset should represent one scheduled stop of one train trip per row.

In [ ]:
schedule = stop_times.merge(
    stops,
    on="stop_id",
    how="left"
)

print(schedule.shape)
schedule.head()

(174124, 17)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
0,1.TA.20-SV1-A-j26-1.1.R,05:27:00,05:27:00,at:44:44900:0:22,1,NaN,0,0,0.00,St.Nikola-Struden Bahnhst,48.233218,14.909141,NaN,NaN,Pat:44:44900,Level 0,22
1,1.TA.20-SV1-A-j26-1.1.R,05:32:00,05:32:00,at:44:44886:0:2,2,NaN,0,0,4601.58,Grein Schiffstation (Donaulände),48.227114,14.856275,NaN,NaN,Pat:44:44886,Level 0,2
2,1.TA.20-SV1-A-j26-1.1.R,05:37:00,05:37:00,at:44:44880:0:32,3,NaN,0,0,5546.31,Grein-Bad Kreuzen Bahnhof,48.220842,14.851451,NaN,NaN,Pat:44:44880,Level 0,Vorplatz
3,1.TA.20-SV1-B-j26-1.1.R,15:41:00,15:41:00,at:44:43649:0:22,1,NaN,0,0,0.00,Windischgarsten Bahnhof,47.715959,14.327428,NaN,NaN,Pat:44:43649,Level -1,Vorplatz
4,1.TA.20-SV1-B-j26-1.1.R,15:47:00,15:47:00,at:44:43640:0:22,2,NaN,0,0,2389.60,Roßleithen Bahnhst,47.724577,14.307953,NaN,NaN,Pat:44:43640,Level 0,22


In [ ]:
schedule["stop_name"].isna().sum()

np.int64(0)

In [ ]:
schedule = schedule.merge(
    trips,
    on="trip_id",
    how="left"
)

print(schedule.shape)
schedule.head()

(174124, 24)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,stop_name,...,parent_station,level_id,platform_code,route_id,service_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,1.TA.20-SV1-A-j26-1.1.R,05:27:00,05:27:00,at:44:44900:0:22,1,NaN,0,0,0.00,St.Nikola-Struden Bahnhst,...,Pat:44:44900,Level 0,22,20-SV1-A-j26-1,TA+eE,20-SV1-A-j26-1.1.R,Grein-Bad Kreuzen Bahnhof,NaN,1,NaN
1,1.TA.20-SV1-A-j26-1.1.R,05:32:00,05:32:00,at:44:44886:0:2,2,NaN,0,0,4601.58,Grein Schiffstation (Donaulände),...,Pat:44:44886,Level 0,2,20-SV1-A-j26-1,TA+eE,20-SV1-A-j26-1.1.R,Grein-Bad Kreuzen Bahnhof,NaN,1,NaN
2,1.TA.20-SV1-A-j26-1.1.R,05:37:00,05:37:00,at:44:44880:0:32,3,NaN,0,0,5546.31,Grein-Bad Kreuzen Bahnhof,...,Pat:44:44880,Level 0,Vorplatz,20-SV1-A-j26-1,TA+eE,20-SV1-A-j26-1.1.R,Grein-Bad Kreuzen Bahnhof,NaN,1,NaN
3,1.TA.20-SV1-B-j26-1.1.R,15:41:00,15:41:00,at:44:43649:0:22,1,NaN,0,0,0.00,Windischgarsten Bahnhof,...,Pat:44:43649,Level -1,Vorplatz,20-SV1-B-j26-1,TA+9f100,20-SV1-B-j26-1.1.R,Klaus an der Pyhrnbahn Bahnhof,NaN,1,NaN
4,1.TA.20-SV1-B-j26-1.1.R,15:47:00,15:47:00,at:44:43640:0:22,2,NaN,0,0,2389.60,Roßleithen Bahnhst,...,Pat:44:43640,Level 0,22,20-SV1-B-j26-1,TA+9f100,20-SV1-B-j26-1.1.R,Klaus an der Pyhrnbahn Bahnhof,NaN,1,NaN


In [ ]:
schedule = schedule.merge(
    routes,
    on="route_id",
    how="left"
)

print(schedule.shape)
schedule.head()

(174124, 28)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,stop_name,...,service_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type
0,1.TA.20-SV1-A-j26-1.1.R,05:27:00,05:27:00,at:44:44900:0:22,1,NaN,0,0,0.00,St.Nikola-Struden Bahnhst,...,TA+eE,20-SV1-A-j26-1.1.R,Grein-Bad Kreuzen Bahnhof,NaN,1,NaN,1,SV133,SV133,3
1,1.TA.20-SV1-A-j26-1.1.R,05:32:00,05:32:00,at:44:44886:0:2,2,NaN,0,0,4601.58,Grein Schiffstation (Donaulände),...,TA+eE,20-SV1-A-j26-1.1.R,Grein-Bad Kreuzen Bahnhof,NaN,1,NaN,1,SV133,SV133,3
2,1.TA.20-SV1-A-j26-1.1.R,05:37:00,05:37:00,at:44:44880:0:32,3,NaN,0,0,5546.31,Grein-Bad Kreuzen Bahnhof,...,TA+eE,20-SV1-A-j26-1.1.R,Grein-Bad Kreuzen Bahnhof,NaN,1,NaN,1,SV133,SV133,3
3,1.TA.20-SV1-B-j26-1.1.R,15:41:00,15:41:00,at:44:43649:0:22,1,NaN,0,0,0.00,Windischgarsten Bahnhof,...,TA+9f100,20-SV1-B-j26-1.1.R,Klaus an der Pyhrnbahn Bahnhof,NaN,1,NaN,1,SV140,SV140,3
4,1.TA.20-SV1-B-j26-1.1.R,15:47:00,15:47:00,at:44:43640:0:22,2,NaN,0,0,2389.60,Roßleithen Bahnhst,...,TA+9f100,20-SV1-B-j26-1.1.R,Klaus an der Pyhrnbahn Bahnhof,NaN,1,NaN,1,SV140,SV140,3


In [ ]:
schedule = schedule[
    [
        "trip_id",
        "route_id",
        "route_short_name",
        "route_long_name",
        "trip_headsign",
        "trip_short_name",
        "direction_id",
        "service_id",
        "stop_id",
        "stop_name",
        "stop_sequence",
        "arrival_time",
        "departure_time",
        "stop_lat",
        "stop_lon",
        "platform_code",
        "shape_dist_traveled"
    ]
].copy()

schedule.head()

,trip_id,route_id,route_short_name,route_long_name,trip_headsign,trip_short_name,direction_id,service_id,stop_id,stop_name,stop_sequence,arrival_time,departure_time,stop_lat,stop_lon,platform_code,shape_dist_traveled
0,1.TA.20-SV1-A-j26-1.1.R,20-SV1-A-j26-1,SV133,SV133,Grein-Bad Kreuzen Bahnhof,NaN,1,TA+eE,at:44:44900:0:22,St.Nikola-Struden Bahnhst,1,05:27:00,05:27:00,48.233218,14.909141,22,0.00
1,1.TA.20-SV1-A-j26-1.1.R,20-SV1-A-j26-1,SV133,SV133,Grein-Bad Kreuzen Bahnhof,NaN,1,TA+eE,at:44:44886:0:2,Grein Schiffstation (Donaulände),2,05:32:00,05:32:00,48.227114,14.856275,2,4601.58
2,1.TA.20-SV1-A-j26-1.1.R,20-SV1-A-j26-1,SV133,SV133,Grein-Bad Kreuzen Bahnhof,NaN,1,TA+eE,at:44:44880:0:32,Grein-Bad Kreuzen Bahnhof,3,05:37:00,05:37:00,48.220842,14.851451,Vorplatz,5546.31
3,1.TA.20-SV1-B-j26-1.1.R,20-SV1-B-j26-1,SV140,SV140,Klaus an der Pyhrnbahn Bahnhof,NaN,1,TA+9f100,at:44:43649:0:22,Windischgarsten Bahnhof,1,15:41:00,15:41:00,47.715959,14.327428,Vorplatz,0.00
4,1.TA.20-SV1-B-j26-1.1.R,20-SV1-B-j26-1,SV140,SV140,Klaus an der Pyhrnbahn Bahnhof,NaN,1,TA+9f100,at:44:43640:0:22,Roßleithen Bahnhst,2,15:47:00,15:47:00,47.724577,14.307953,22,2389.60


In [ ]:
schedule = schedule.sort_values(
    by=["trip_id", "stop_sequence"]
).reset_index(drop=True)

schedule.head(20)

,trip_id,route_id,route_short_name,route_long_name,trip_headsign,trip_short_name,direction_id,service_id,stop_id,stop_name,stop_sequence,arrival_time,departure_time,stop_lat,stop_lon,platform_code,shape_dist_traveled
0,1.TA.1-80-j26-1.1.H,1-80-j26-1,80,80,Marchegg Bahnhof,S 2590,0,TA+h0,at:49:1349:0:10,Wien Hauptbahnhof,1,01:21:00,01:21:00,48.184451,16.378345,12,0.00
1,1.TA.1-80-j26-1.1.H,1-80-j26-1,80,80,Marchegg Bahnhof,S 2590,0,TA+h0,at:49:1260:0:15,Wien Simmering,2,01:25:00,01:26:00,48.169560,16.418625,2,3822.47
2,1.TA.1-80-j26-1.1.H,1-80-j26-1,80,80,Marchegg Bahnhof,S 2590,0,TA+h0,at:49:465:0:3,Wien Haidestraße,3,01:28:00,01:28:00,48.178629,16.426225,1,4987.88
3,1.TA.1-80-j26-1.1.H,1-80-j26-1,80,80,Marchegg Bahnhof,S 2590,0,TA+h0,at:49:1301:0:4,Wien Praterkai,4,01:31:00,01:32:00,48.199530,16.440984,2,7564.55
4,1.TA.1-80-j26-1.1.H,1-80-j26-1,80,80,Marchegg Bahnhof,S 2590,0,TA+h0,at:49:1299:0:5,Wien Stadlau,5,01:34:00,01:35:00,48.218693,16.448117,1,9964.42
5,1.TA.1-80-j26-1.1.H,1-80-j26-1,80,80,Marchegg Bahnhof,S 2590,0,TA+h0,at:49:288:0:15,Wien Erzherzog-Karl-Straße,6,01:38:00,01:38:00,48.230076,16.453740,4,11300.65
6,1.TA.1-80-j26-1.1.H,1-80-j26-1,80,80,Marchegg Bahnhof,S 2590,0,TA+h0,at:49:532:0:6,Wien Hirschstetten,7,01:40:00,01:40:00,48.232745,16.469111,2,12513.09
7,1.TA.1-80-j26-1.1.H,1-80-j26-1,80,80,Marchegg Bahnhof,S 2590,0,TA+h0,at:49:910:0:7,Wien Aspern Nord,8,01:43:00,01:44:00,48.234785,16.504729,2,15162.44
8,1.TA.1-80-j26-1.1.H,1-80-j26-1,80,80,Marchegg Bahnhof,S 2590,0,TA+h0,at:43:4613:0:3,Raasdorf Bahnhof,9,01:47:00,01:48:00,48.239111,16.583044,2,20981.90
9,1.TA.1-80-j26-1.1.H,1-80-j26-1,80,80,Marchegg Bahnhof,S 2590,0,TA+h0,at:43:3527:0:3,Glinzendorf Bahnhof,10,01:51:00,01:52:00,48.242138,16.640240,2,25231.14


In [ ]:
example_trip_id = schedule["trip_id"].iloc[0]

schedule[schedule["trip_id"] == example_trip_id][
    [
        "trip_id",
        "route_short_name",
        "trip_headsign",
        "stop_sequence",
        "stop_name",
        "arrival_time",
        "departure_time"
    ]
]

,trip_id,route_short_name,trip_headsign,stop_sequence,stop_name,arrival_time,departure_time
0,1.TA.1-80-j26-1.1.H,80,Marchegg Bahnhof,1,Wien Hauptbahnhof,01:21:00,01:21:00
1,1.TA.1-80-j26-1.1.H,80,Marchegg Bahnhof,2,Wien Simmering,01:25:00,01:26:00
2,1.TA.1-80-j26-1.1.H,80,Marchegg Bahnhof,3,Wien Haidestraße,01:28:00,01:28:00
3,1.TA.1-80-j26-1.1.H,80,Marchegg Bahnhof,4,Wien Praterkai,01:31:00,01:32:00
4,1.TA.1-80-j26-1.1.H,80,Marchegg Bahnhof,5,Wien Stadlau,01:34:00,01:35:00
5,1.TA.1-80-j26-1.1.H,80,Marchegg Bahnhof,6,Wien Erzherzog-Karl-Straße,01:38:00,01:38:00
6,1.TA.1-80-j26-1.1.H,80,Marchegg Bahnhof,7,Wien Hirschstetten,01:40:00,01:40:00
7,1.TA.1-80-j26-1.1.H,80,Marchegg Bahnhof,8,Wien Aspern Nord,01:43:00,01:44:00
8,1.TA.1-80-j26-1.1.H,80,Marchegg Bahnhof,9,Raasdorf Bahnhof,01:47:00,01:48:00
9,1.TA.1-80-j26-1.1.H,80,Marchegg Bahnhof,10,Glinzendorf Bahnhof,01:51:00,01:52:00


## 7. Convert time values

The arrival and departure times are stored as text.

We convert them into seconds to make later calculations easier.

In [ ]:
def gtfs_time_to_seconds(time_str):
    if pd.isna(time_str):
        return None
    
    hours, minutes, seconds = map(int, time_str.split(":"))
    return hours * 3600 + minutes * 60 + seconds

In [ ]:
schedule["arrival_seconds"] = schedule["arrival_time"].apply(gtfs_time_to_seconds)
schedule["departure_seconds"] = schedule["departure_time"].apply(gtfs_time_to_seconds)

schedule[
    [
        "arrival_time",
        "arrival_seconds",
        "departure_time",
        "departure_seconds"
    ]
].head()

,arrival_time,arrival_seconds,departure_time,departure_seconds
0,01:21:00,4860,01:21:00,4860
1,01:25:00,5100,01:26:00,5160
2,01:28:00,5280,01:28:00,5280
3,01:31:00,5460,01:32:00,5520
4,01:34:00,5640,01:35:00,5700


## 8. Extract hour information

We create new columns with the departure hour.

This will help us analyze delays by time of day later.

In [ ]:
schedule["scheduled_hour"] = schedule["departure_seconds"] // 3600
schedule["scheduled_hour_mod24"] = schedule["scheduled_hour"] % 24

schedule[
    [
        "departure_time",
        "scheduled_hour",
        "scheduled_hour_mod24"
    ]
].head()

,departure_time,scheduled_hour,scheduled_hour_mod24
0,01:21:00,1,1
1,01:26:00,1,1
2,01:28:00,1,1
3,01:32:00,1,1
4,01:35:00,1,1


## 9. Create time categories

We group departures into simple time categories:

- morning peak
- midday
- evening peak
- evening
- night

These categories will be useful for later analysis and visualizations.

In [ ]:
def get_time_of_day(hour):
    if pd.isna(hour):
        return "unknown"
    elif 6 <= hour < 9:
        return "morning_peak"
    elif 9 <= hour < 16:
        return "midday"
    elif 16 <= hour < 19:
        return "evening_peak"
    elif 19 <= hour < 24:
        return "evening"
    else:
        return "night"

schedule["time_of_day"] = schedule["scheduled_hour_mod24"].apply(get_time_of_day)

schedule[["departure_time", "scheduled_hour_mod24", "time_of_day"]].head(20)

,departure_time,scheduled_hour_mod24,time_of_day
0,01:21:00,1,night
1,01:26:00,1,night
2,01:28:00,1,night
3,01:32:00,1,night
4,01:35:00,1,night
5,01:38:00,1,night
6,01:40:00,1,night
7,01:44:00,1,night
8,01:48:00,1,night
9,01:52:00,1,night


## 11. Check missing values

Finally, we check the merged dataset for missing values.

In [ ]:
print("Final schedule shape:", schedule.shape)

schedule.isna().sum().sort_values(ascending=False)

Final schedule shape: (174124, 22)


trip_short_name         1346
platform_code            582
route_long_name          534
trip_id                    0
departure_time             0
scheduled_hour_mod24       0
scheduled_hour             0
departure_seconds          0
arrival_seconds            0
shape_dist_traveled        0
stop_lon                   0
stop_lat                   0
arrival_time               0
route_id                   0
stop_sequence              0
stop_name                  0
stop_id                    0
service_id                 0
direction_id               0
trip_headsign              0
route_short_name           0
time_of_day                0
dtype: int64

In [ ]:
schedule[schedule["route_long_name"].isna()][
    ["route_id", "route_short_name", "route_long_name"]
].drop_duplicates().head(20)

,route_id,route_short_name,route_long_name
447,12-A-1-j26-1,A-1,NaN
577,12-CH2-j26-1,CH2,NaN
609,12-D19-j26-1,D19,NaN
645,13-A-2-j26-1,A-2,NaN
21319,13-A-1-j26-1,A-1,NaN


In [ ]:
schedule["route_name"] = schedule["route_long_name"].fillna(schedule["route_short_name"])

In [ ]:
schedule[["route_short_name", "route_long_name", "route_name"]].head(20)

,route_short_name,route_long_name,route_name
0,80,80,80
1,80,80,80
2,80,80,80
3,80,80,80
4,80,80,80
5,80,80,80
6,80,80,80
7,80,80,80
8,80,80,80
9,80,80,80


In [ ]:
from pathlib import Path

processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

schedule.to_csv(processed_path / "gtfs_schedule_stop_level.csv", index=False)

print("Saved to:", processed_path / "gtfs_schedule_stop_level.csv")

Saved to: ../data/processed/gtfs_schedule_stop_level.csv


# Summary

In this notebook, we:

- loaded the GTFS data
- explored the main tables
- merged the datasets
- created time-related features
- checked missing values

The dataset is now ready for the next step: joining it with the actual train performance data.